In [ ]:
from functools import partial
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

In [ ]:
def sample_nngs(k, p, N, rng, runs):
    # Fase 1
    # ngood = int(np.floor(p * k))
    # d = rng.binomial(n=ngood, p=p, size=runs)
    d = int(np.floor(p * k)) * np.ones(runs, dtype=int)
    
    # Fase 2
    ngood = k - d
    nbad = int(np.floor((1.0 - p) * (N - k)))
    m = rng.hypergeometric(ngood=ngood, nbad=nbad, nsample=ngood)

    t = d + m

    nngs = t / (2.0 * k - t)
    return nngs

def simulate_run(k, p, N, runs, seed):
    rng = np.random.default_rng(seed)
    nngss = sample_nngs(k, p, N, rng, runs)
    return (k, p, np.mean(nngss))

def do_experiment(ks, ps, N, runs):
    do_run = delayed(partial(simulate_run, N=N, runs=runs))
    with Parallel(n_jobs=-1, backend='loky') as parallel:
        results = parallel(
            do_run(k=k, p=p, seed=seed) for seed, (k, p) in enumerate(product(ks, ps)))
    return results

In [ ]:
runs = 10000
N = 10000
ks = np.arange(1, N + 1, 1, dtype=int)
ps = np.linspace(0.0, 1.0, 11, endpoint=True)

In [ ]:
nngss = do_experiment(ks, ps, N, runs)

In [ ]:
cols = ["k", "p", "nngs"]
df_nngs = pd.DataFrame(nngss, columns=cols).pivot(index="k", columns="p", values="nngs")

In [ ]:
nngs_lower_bound = ks / (2*N - ks)

In [ ]:
df_nngs_mean_normalized = (df_nngs - nngs_lower_bound[:, np.newaxis]) / (1.0 - nngs_lower_bound[:, np.newaxis])

In [ ]:
plt.figure(figsize=(10, 6))
df_nngs.plot(ax=plt.gca(), legend=False, title="NNGS unnormalized")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
df_nngs_mean_normalized.plot(ax=plt.gca(), legend=False, title="NNGS normalized")
plt.show()